# Circular-Shift Permutation GLM for Fiber Photometry

This notebook demonstrates the `glm-photometry` package for fitting generalized linear models to fiber photometry peri-event time series data. The approach uses circular-shift permutation testing (Hjort et al.) for non-parametric significance evaluation.

**Overview of the pipeline:**
1. Load peri-event photometry data (HDF5 or Excel)
2. Build design matrices with task-relevant predictors
3. Fit the GLM and compute delta-R2 for each predictor
4. Generate a null distribution via circular-shift permutations
5. Determine significance and visualize results

**Predictor chunks tested:**
- **Solution identity** (dummy-coded vs. water reference)
- **Lick count** (z-scored, continuous)
- **Lick x solution interaction** (optional)
- **Trial number** (linear drift within session)
- **Session** (nuisance regressors)

## 0. Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# If running from the notebooks/ directory, add the parent to the path
sys.path.insert(0, str(Path("..").resolve()))

import glm_photometry as glm

print(f"glm-photometry version: {glm.__version__}")
print(f"NumPy: {np.__version__}, Pandas: {pd.__version__}")

## 1. Configuration

Set the path to your data file and analysis parameters below. The data should be either:
- An **HDF5 file** (`.h5`) with a `streams_peth` group containing columns like `subject`, `channel_id`, `blockname`, `solution`, `time_rel`, `lick_count`, and the signal column.
- An **Excel file** (`.xlsx`) with the same column structure.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────
# Path to your peri-event data file (.h5 or .xlsx)
DATA_PATH = Path("../data/streams_peth_multi_mix_all.h5")

# Signal column to use as the response variable
SIGNAL_COL = "delta_signal_poly_zscore_blsub"

# GLM parameters
N_SHIFTS = 5000        # number of circular-shift permutations
ALPHA = 0.01           # significance threshold
REFERENCE = "water"    # reference solution for dummy coding

# Output directory for figures
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"Data path: {DATA_PATH}")
print(f"Signal column: {SIGNAL_COL}")
print(f"Permutations: {N_SHIFTS}, Alpha: {ALPHA}")

## 2. Load Data

In [ ]:
# Load data -- use load_h5_data for .h5 files, load_fp_data for .xlsx
if DATA_PATH.suffix == ".h5":
    df = glm.load_h5_data(str(DATA_PATH))
else:
    df = glm.load_fp_data(str(DATA_PATH))

subjects = sorted(df["subject"].unique())
solutions = sorted(df["solution"].unique())
channels = sorted(df["channel_id"].unique())

print(f"Loaded {len(df):,} rows")
print(f"Subjects: {len(subjects)}")
print(f"Solutions: {solutions}")
print(f"Channel IDs: {channels}")
print(f"\nFirst few rows:")
df.head()

## 3. Run GLM for a Single Subject

Before running the full batch, it is useful to examine the GLM for one subject to verify the design matrix structure and inspect the results.

In [ ]:
# Pick one subject to demo
demo_subject = subjects[0]
print(f"Demo subject: {demo_subject}")

df_subj = df[df["subject"] == demo_subject]
channel_id = df_subj["channel_id"].iloc[0]
sessions = sorted(df_subj["blockname"].unique())
print(f"Channel: {channel_id}, Sessions: {len(sessions)}")
print(f"Trials per session: ", end="")
for sess in sessions:
    n = df_subj[df_subj["blockname"] == sess].groupby(
        ["solution", "event_number_solution"]).ngroups
    print(f"{n}", end=" ")
print()

### 3.1 Build the Design Matrix

The design matrix is constructed per-subject, concatenating all sessions. Each predictor group (\"chunk\") is stored separately to enable leave-one-out F-testing.

In [ ]:
# Prepare data for one subject
(Y, trial_info_list, time_rel, T_trial,
 session_lengths, session_trial_counts,
 boundaries, sess_names) = glm.prepare_channel_data(
    df_subj, channel_id, signal_col=SIGNAL_COL
)

print(f"Signal length: {len(Y)} timepoints")
print(f"Timepoints per trial: {T_trial}")
print(f"Time range: [{time_rel[0]:.2f}, {time_rel[-1]:.2f}] s")
print(f"Session boundaries: {boundaries}")

# Build design matrix (omnibus solution predictor)
X_list, chunk_names = glm.build_design_matrix(
    trial_info_list, T_trial, session_lengths, solutions=solutions
)

print(f"\nDesign matrix chunks:")
for i, (name, X_chunk) in enumerate(zip(chunk_names, X_list)):
    print(f"  [{i}] {name:20s}  shape={X_chunk.shape}")

In [ ]:
# Visualize the design matrix
X_full = np.column_stack([np.ones((len(Y), 1))] + X_list)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(X_full[:500, :].T, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xlabel("Timepoint (first 500)")
ax.set_ylabel("Predictor column")
ax.set_title("Design Matrix (first 500 timepoints)")
plt.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

### 3.2 Run Permutation GLM (Single Subject)

The GLM is fit once on the observed data to get the F-statistic and delta-R2 for each predictor chunk. Then the signal is circularly shifted within each session boundary to build a null distribution. P-values are computed as the fraction of null F-statistics that exceed the observed value.

In [ ]:
%%time
# Run the circular-shift permutation GLM
# For a quick test, use fewer shifts (e.g., 500). For publication, use 5000.
demo_n_shifts = 1000  # reduce for notebook demo speed

result = glm.run_permutation_glm(
    Y, X_list, boundaries,
    n_shifts=demo_n_shifts, alpha=ALPHA, seed=42, verbose=False
)

print(f"Full model R2: {result['r2_full']:.4f}\n")
print(f"{'Predictor':<22s} {'dR2':>8s} {'F-obs':>8s} {'p-value':>8s} {'Sig':>5s}")
print("-" * 55)
for k, name in enumerate(chunk_names):
    sig = "*" if result["significant"][k] else ""
    print(f"{name:<22s} {result['delta_r2_obs'][k]:8.4f} "
          f"{result['f_obs'][k]:8.2f} {result['pvals'][k]:8.4f} {sig:>5s}")

In [ ]:
# Visualize null distributions vs. observed F-statistics
task_idx = [i for i, n in enumerate(chunk_names) if n != "session"]
n_task = len(task_idx)
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, n_task, figsize=(3.5 * n_task, 3))
if n_task == 1:
    axes = [axes]

for pi, ki in enumerate(task_idx):
    ax = axes[pi]
    ax.hist(result["f_null"][:, ki], bins=50, color=colors[pi],
            alpha=0.7, edgecolor="white", linewidth=0.3, density=True)
    ax.axvline(result["f_obs"][ki], color="red", linewidth=2,
               linestyle="--", label=f"Observed F={result['f_obs'][ki]:.1f}")
    ax.set_xlabel("F-statistic")
    ax.set_ylabel("Density")
    ax.set_title(f"{chunk_names[ki]}\np={result['pvals'][ki]:.4f}")
    ax.legend(fontsize=7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Null distributions vs. observed F-statistics", fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

### 3.3 Time-Resolved GLM

Slide a 1.0 s window (0.25 s step) across the peri-event period to visualize when each predictor becomes significant.

In [ ]:
%%time
tr_result = glm.run_timeresolved(
    Y, trial_info_list, time_rel, T_trial,
    session_trial_counts,
    win_size=1.0, win_step=0.25,
    n_shifts=demo_n_shifts, alpha=ALPHA, seed=42, verbose=False
)

print(f"Windows: {len(tr_result['win_centers'])}")
print(f"Chunk names: {tr_result['chunk_names']}")

In [ ]:
# Plot time-resolved delta-R2
fig, ax = plt.subplots(figsize=(8, 3.5))
tr_names = tr_result["chunk_names"]
tr_task_idx = [i for i, n in enumerate(tr_names) if n != "session"]

for pi, ki in enumerate(tr_task_idx):
    ax.plot(tr_result["win_centers"], tr_result["delta_r2"][:, ki],
            color=colors[pi], linewidth=1.5, label=tr_names[ki])
    # Mark significant windows
    sig_mask = tr_result["significant"][:, ki]
    ax.scatter(tr_result["win_centers"][sig_mask],
               tr_result["delta_r2"][sig_mask, ki],
               color=colors[pi], s=20, zorder=5)

ax.axvline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Time from access onset (s)")
ax.set_ylabel("$\\Delta R^2$")
ax.set_title(f"Time-resolved encoding: {demo_subject} ({channel_id})")
ax.legend(fontsize=8, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Batch Analysis: All Subjects

Run the GLM across all subjects and collect results for group-level statistics.

In [ ]:
%%time
import time as _time

all_results = []
for si, subj in enumerate(subjects):
    ch = df.loc[df["subject"] == subj, "channel_id"].iloc[0]
    print(f"[{si+1}/{len(subjects)}] {subj} ({ch})...", end=" ", flush=True)
    t0 = _time.time()
    
    res = glm.run_subject_glm(df, subj,
                               n_shifts=N_SHIFTS, alpha=ALPHA, seed=42 + si)
    all_results.append(res)
    
    sig_preds = [res["chunk_names"][k] for k in range(len(res["chunk_names"]))
                 if res["full_glm"]["significant"][k]
                 and res["chunk_names"][k] != "session"]
    elapsed = _time.time() - t0
    print(f"{elapsed:.1f}s  R2={res['full_glm']['r2_full']:.3f}  sig: {sig_preds}")

print(f"\nDone. {len(all_results)} subjects processed.")

## 5. Group-Level Summary

Aggregate results across subjects, grouped by population (channel_id).

In [ ]:
from scipy.stats import kruskal, mannwhitneyu

# Organize by population
POP_MAP = {"cbln4": "Cbln4", "crhbp": "Crhbp", "pnoc": "Pnoc"}  # adjust to your data
POP_COLORS = {"Cbln4": "#4C72B0", "Crhbp": "#DD8452", "Pnoc": "#55A868"}

pops = {}
for r in all_results:
    pop = POP_MAP.get(r["channel_id"], r["channel_id"])
    pops.setdefault(pop, []).append(r)

chunk_names_ref = all_results[0]["chunk_names"]
task_idx = [i for i, n in enumerate(chunk_names_ref) if n != "session"]

# Summary table
rows = []
for ki in task_idx:
    name = chunk_names_ref[ki]
    for pop_name in sorted(pops.keys()):
        rr = pops[pop_name]
        dr2s = [r["full_glm"]["delta_r2_obs"][ki] for r in rr]
        sigs = [r["full_glm"]["significant"][ki] for r in rr]
        rows.append({
            "Predictor": name,
            "Population": pop_name,
            "n": len(rr),
            "dR2_mean": np.mean(dr2s),
            "dR2_std": np.std(dr2s),
            "pct_sig": 100 * np.mean(sigs),
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Group comparison: Kruskal-Wallis across populations
pop_names = sorted(pops.keys())
print(f"{'Predictor':<22s} {'H':>8s} {'p':>8s} {'Sig':>5s}")
print("-" * 50)
for ki in task_idx:
    name = chunk_names_ref[ki]
    groups = []
    for pop_name in pop_names:
        dr2s = [r["full_glm"]["delta_r2_obs"][ki] for r in pops[pop_name]]
        groups.append(dr2s)
    if all(len(g) >= 2 for g in groups):
        H, p = kruskal(*groups)
        sig = "*" if p < 0.05 else ""
        print(f"{name:<22s} {H:8.2f} {p:8.4f} {sig:>5s}")
        # Pairwise Mann-Whitney U if significant
        if p < 0.05:
            for i in range(len(pop_names)):
                for j in range(i + 1, len(pop_names)):
                    U, pw = mannwhitneyu(groups[i], groups[j], alternative="two-sided")
                    print(f"  {pop_names[i]} vs {pop_names[j]}: U={U:.0f}, p={pw:.4f}")

## 6. Group Visualization

In [ ]:
# Delta-R2 bar plot with individual subjects
fig, axes = plt.subplots(1, len(task_idx), figsize=(3.5 * len(task_idx), 4), sharey=True)
if len(task_idx) == 1:
    axes = [axes]

for mi, ki in enumerate(task_idx):
    ax = axes[mi]
    name = chunk_names_ref[ki]
    for pi, pop_name in enumerate(pop_names):
        vals = np.array([r["full_glm"]["delta_r2_obs"][ki] for r in pops[pop_name]])
        color = POP_COLORS.get(pop_name, f"C{pi}")
        
        # Individual points with jitter
        jitter = np.random.default_rng(42 + pi).normal(0, 0.08, len(vals))
        ax.scatter(np.full_like(vals, pi) + jitter, vals,
                   color=color, s=25, alpha=0.6, edgecolors="black", linewidth=0.3)
        # Mean +/- SEM
        ax.errorbar(pi, np.mean(vals), yerr=np.std(vals) / np.sqrt(len(vals)),
                     fmt="o", color="black", markersize=6, capsize=4, linewidth=1.5)
        # % significant
        pct = 100 * np.mean([r["full_glm"]["significant"][ki] for r in pops[pop_name]])
        ax.text(pi, ax.get_ylim()[1] if mi > 0 else np.max(vals) * 1.15,
                f"{pct:.0f}%", ha="center", fontsize=7, color=color)
    
    ax.set_xticks(range(len(pop_names)))
    ax.set_xticklabels(pop_names, fontsize=9)
    ax.set_title(name, fontsize=10, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel("$\\Delta R^2$")
fig.suptitle("Encoding strength by population", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "glm_group_dr2_bar.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Per-Solution GLM (Optional)

Instead of testing an omnibus solution effect, decompose into individual solution contrasts (sucrose vs. water, NaCl vs. water, dry vs. water).

In [ ]:
# Run per-solution GLM for the demo subject
(Y_ps, trial_info_list_ps, time_rel_ps, T_trial_ps,
 session_lengths_ps, session_trial_counts_ps,
 boundaries_ps, _) = glm.prepare_channel_data(
    df_subj, channel_id, signal_col=SIGNAL_COL
)

# Build per-solution design matrix
X_ps_list, ps_chunk_names = glm.build_per_solution_dm(
    trial_info_list_ps, T_trial_ps, session_lengths_ps, solutions=solutions
)

print("Per-solution design matrix chunks:")
for i, (name, X_chunk) in enumerate(zip(ps_chunk_names, X_ps_list)):
    print(f"  [{i}] {name:20s}  shape={X_chunk.shape}")

# Run GLM
ps_result = glm.run_permutation_glm(
    Y_ps, X_ps_list, boundaries_ps,
    n_shifts=demo_n_shifts, alpha=ALPHA, seed=42, verbose=False
)

print(f"\nFull model R2: {ps_result['r2_full']:.4f}\n")
for k, name in enumerate(ps_chunk_names):
    sig = "*" if ps_result["significant"][k] else ""
    print(f"{name:<22s} dR2={ps_result['delta_r2_obs'][k]:.4f}  "
          f"p={ps_result['pvals'][k]:.4f} {sig}")

## 8. Export Results

Save all results to CSV for downstream analysis or manuscript preparation.

In [ ]:
# Build a tidy DataFrame of all results
export_rows = []
for r in all_results:
    pop = POP_MAP.get(r["channel_id"], r["channel_id"])
    for ki in task_idx:
        export_rows.append({
            "subject": r["subject"],
            "channel_id": r["channel_id"],
            "population": pop,
            "predictor": r["chunk_names"][ki],
            "delta_r2": r["full_glm"]["delta_r2_obs"][ki],
            "f_obs": r["full_glm"]["f_obs"][ki],
            "p_value": r["full_glm"]["pvals"][ki],
            "significant": r["full_glm"]["significant"][ki],
            "r2_full": r["full_glm"]["r2_full"],
        })

results_df = pd.DataFrame(export_rows)
csv_path = OUTPUT_DIR / "glm_results_all_subjects.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved {len(results_df)} rows to {csv_path}")
results_df.head(10)